# Q-RESILIENCE AI — Exploration Notebook
### Team Ctrl_AI (VTU-QFF-HC-0025) · VTU Qiskit Fall Fest 2026

This notebook demonstrates the end-to-end quantum machine learning pipeline for microgrid anomaly detection.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

import qiskit
from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC

print(f"Qiskit Version: {qiskit.__version__}")

In [ ]:
# 1. Load Microgrid Telemetry
df = pd.read_csv('../data/microgrid_telemetry.csv')
print(f"Loaded {len(df)} telemetry samples")
df.head()

In [ ]:
# 2. Preprocess features and scale to [0, 2pi]
feature_cols = ['demand_kw', 'solar_kw', 'voltage_dev_v', 'frequency_dev_hz']
X = df[feature_cols].values
y = df['is_abnormal'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
scaler = MinMaxScaler(feature_range=(0, 2 * np.pi))
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f"Train shape: {X_train_scaled.shape}, Test shape: {X_test_scaled.shape}")

In [ ]:
# 3. Construct Qiskit ZZFeatureMap
feature_map = ZZFeatureMap(feature_dimension=4, reps=2, entanglement='linear')
feature_map.decompose().draw('mpl')

In [ ]:
# 4. Classical SVC Baseline
classical_svc = SVC(C=1.0, kernel='rbf', gamma='scale', random_state=42)
classical_svc.fit(X_train_scaled, y_train)
c_preds = classical_svc.predict(X_test_scaled)
print("Classical SVC Report:")
print(classification_report(y_test, c_preds))

In [ ]:
# 5. Quantum QSVC with Fidelity Quantum Kernel
kernel = FidelityQuantumKernel(feature_map=feature_map)
qsvc = QSVC(quantum_kernel=kernel, C=1.0)
qsvc.fit(X_train_scaled, y_train)
q_preds = qsvc.predict(X_test_scaled)
print("Quantum QSVC Report:")
print(classification_report(y_test, q_preds))